# Run GLiNER on sample documents

Take HTML from `data/samples/`, strip to text, run the same `GlinerBackend` as the tender preprocess stage, and inspect labeled spans.

Full pipeline: `python scripts/run_preprocess.py --path tender --stage gliner …`  
Deep walkthrough: [`../notebooks/preprocess_stage_inspect.ipynb`](../notebooks/preprocess_stage_inspect.ipynb)

**How to use this notebook:** change the knobs in §1–§3 for different files / use cases (see **Tuning guide** below). Re-run from that cell downward.


## Tuning guide (files + use cases)

GLiNER is **zero-shot**: you steer it with (1) which file you feed, (2) which label strings you ask for, (3) how much text / how strict the score cut is.

### 1. Pick the right sample folder (`FOLDER` in §1)

| Use case | Set `FOLDER` to | Why |
|---|---|---|
| Tender **offer terms** (price, expire, CUSIP) | `offer_to_purchase` | Long narrative OTP — best for TO labels |
| Tender **cover / who-is-who** | `to_t`, `to_i`, or `schedule_to` | Short cover sheets |
| Target board recommendation | `14d9` | Different vocabulary; keep org/date labels, drop tender-only labels if noisy |
| Amendments / what changed | `amendments` | Often shorter; focus on price/date deltas |
| 8-K **body** (items, short disclosures) | `8k` | Short; use fewer, broader labels |
| 8-K **exhibits** (agreements, press) | `8k_exhibits` | Longer contracts/press; pick labels for parties, dates, amounts |

Also filter by year / entity in §1 if you want a specific filing:
```python
YEAR = 2024
ENTITY_SUBSTR = "Cerner"
```

### 2. Change labels for the use case (`USE_CASE` / `LABELS` in §2)

Labels are plain English phrases GLiNER looks for — **not** regex. Prefer **value-shaped** names (`offer_price_amount`) over heading-like ones (`Offer Price`).

| Use case | Set `USE_CASE` |
|---|---|
| Full tender extract (default) | `tender_full` → `GLINER_LABELS_TO` |
| Cover / parties only | `tender_cover` |
| Summary / FAQ slice | `tender_summary` |
| 8-K / M&A announcement | `eight_k` |
| IDs only | `ids_only` |
| Your own list | `custom` (edit the list in §2) |

Fewer labels → faster + less noise. Too many vague labels → headings get tagged as values.

### 3. Score / window / length knobs (§2–§3)

| Knob | Where | Effect |
|---|---|---|
| `THRESHOLD` | §2 | Higher (e.g. `0.55`) = fewer, cleaner spans; lower (e.g. `0.3`) = more recall |
| `MAX_WINDOWS` | §2 | How many ~900-char chunks to scan; raise for long OTPs |
| `WINDOW_CHARS` / `OVERLAP_CHARS` | §2 | Chunk size / overlap; defaults are fine for most demos |
| `MAX_CHARS` | §3 | Truncates demo input; raise to read deeper (slower) |
| `MODEL_ID` | §2 | `gliner_medium-v2.1` vs `urchade/gliner_large-v2.1` |
| `N` / `MAX_CHARS_BATCH` | §5 | Batch smoke-test size |

### 4. Quick recipes

- **Noise / headings tagged as price:** raise `THRESHOLD` to `0.5+`; keep value-oriented labels only.
- **Missed price deep in OTP:** raise `MAX_CHARS` and `MAX_WINDOWS`.
- **8-K empty / sparse spans:** set `FOLDER="8k"` or `"8k_exhibits"`, `USE_CASE="eight_k"`, maybe lower `THRESHOLD` slightly.
- **Compare two filings:** set `ENTITY_SUBSTR`, run §3; change entity; run §3 again.

Production note: the real pipeline runs GLiNER **per segment** (not one truncated blob), then ranks spans into a Concise Rep.


## 0. Setup

In [ ]:
from __future__ import annotations

import html as html_lib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path("..").resolve() if Path.cwd().name == "preprocess" else Path(".").resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from eda.documents import html_to_text
from src.preprocess.backends import GlinerBackend

SAMPLES = ROOT / "data" / "samples"
MANIFEST = SAMPLES / "manifest.csv"
assert MANIFEST.exists(), f"Missing {MANIFEST} — run: python dataloader/download_samples.py"

mf = pd.read_csv(MANIFEST)
print(mf.groupby(["folder", "doc_role"], dropna=False).size().to_string())


## 1. Pick a document

Change `FOLDER` (and optional filters) for the file type / use case — see **Tuning guide**.


In [ ]:
# --- knobs: which file ---
FOLDER = "offer_to_purchase"
# tender terms: offer_to_purchase | cover: to_t / to_i / schedule_to
# board: 14d9 | patches: amendments | short current report: 8k | exhibits: 8k_exhibits

YEAR = None       # e.g. 2024 to restrict; None = any
ENTITY_SUBSTR = None  # e.g. "Cerner"; None = any

candidates = mf[(mf.folder == FOLDER) & mf.html_path.notna() & (mf.status == "ok")].copy()
if YEAR is not None:
    candidates = candidates[candidates.year == YEAR]
if ENTITY_SUBSTR:
    candidates = candidates[candidates.entity_name.fillna("").str.contains(ENTITY_SUBSTR, case=False)]
assert len(candidates), f"no HTML rows for folder={FOLDER!r} year={YEAR} entity={ENTITY_SUBSTR!r}"
row = candidates.sort_values("year").iloc[0]

html_path = Path(row.html_path)
assert html_path.exists(), html_path

raw_html = html_path.read_text(errors="replace")
text = html_to_text(raw_html)
print(f"{row.folder} | {row.form} | {row.get('file_type')} | {row.entity_name}")
print(f"file: {html_path.name}")
print(f"chars={len(text):,}  words≈{len(text.split()):,}")
print("--- preview ---")
print(text[:1200], "…" if len(text) > 1200 else "")


## 2. Load GLiNER + choose labels / model knobs

Same backend as tender preprocess. First run downloads weights.  
Swap `USE_CASE` preset or edit `LABELS` / `threshold` / `max_windows` — see **Tuning guide**.


In [ ]:
from src.preprocess.contracts import GLINER_LABELS_TO, LABELS_COVER, LABELS_SUMMARY_FAQ

MODEL_ID = "urchade/gliner_medium-v2.1"
# heavier / often better: "urchade/gliner_large-v2.1"

# --- knobs: use-case → label set ---
USE_CASE = "tender_full"  # tender_full | tender_cover | tender_summary | eight_k | ids_only | custom

PRESETS = {
    "tender_full": list(GLINER_LABELS_TO),
    "tender_cover": list(LABELS_COVER),
    "tender_summary": list(LABELS_SUMMARY_FAQ),
    "eight_k": [
        "acquiring_organization",
        "target_organization",
        "deal_value_amount",
        "effective_datetime",
        "agreement_name",
        "ticker_symbol",
    ],
    "ids_only": ["cusip_code", "isin_code", "ticker_symbol"],
    "custom": [
        # edit freely — plain-English, value-oriented names work best
        "offer_price_amount",
        "expiration_datetime",
        "target_organization",
    ],
}
LABELS = PRESETS[USE_CASE]

# --- knobs: precision / coverage ---
THRESHOLD = 0.4     # try 0.55 if noisy; 0.3 if missing spans
MAX_WINDOWS = 6     # raise for long OTPs (e.g. 12); lower for speed
WINDOW_CHARS = 900
OVERLAP_CHARS = 150

backend = GlinerBackend(
    model_id=MODEL_ID,
    max_windows=MAX_WINDOWS,
    window_chars=WINDOW_CHARS,
    overlap_chars=OVERLAP_CHARS,
    threshold=THRESHOLD,
)
assert backend.available, f"GLiNER unavailable: {backend._load_error}"
print("model:", backend.model_id)
print("use_case:", USE_CASE)
print("threshold:", THRESHOLD, "max_windows:", MAX_WINDOWS)
print("labels:", ", ".join(LABELS))


## 3. Predict spans

`MAX_CHARS` truncates for a fast demo. Raise it (and `MAX_WINDOWS` in §2) to reach terms deeper in long OTPs.


In [ ]:
# --- knobs: how much of the doc to score ---
MAX_CHARS = 12_000  # OTP: try 30_000+; 8-K body: 8_000 is usually enough
run_text = text[:MAX_CHARS]

spans = backend.predict(run_text, LABELS)
span_df = (
    pd.DataFrame(spans)
    .sort_values(["label", "score"], ascending=[True, False])
    .reset_index(drop=True)
)
print(f"spans={len(span_df)} on {len(run_text):,} / {len(text):,} chars")
if span_df.empty:
    print("No spans — try another FOLDER/doc, lower THRESHOLD, or different USE_CASE labels")
else:
    display(span_df.head(40))
    print("\nby label:")
    display(
        span_df.groupby("label")
        .agg(n=("text", "size"), best=("score", "max"))
        .sort_values("n", ascending=False)
    )


## 4. Highlight top spans in context

In [ ]:
def highlight(text: str, spans: pd.DataFrame, *, top_n: int = 15) -> HTML:
    if spans is None or spans.empty:
        return HTML("<pre>(no spans)</pre>")
    top = spans.sort_values("score", ascending=False).head(top_n)
    html_parts = []
    cursor = 0
    for sp in sorted(top.to_dict("records"), key=lambda r: int(r["start"])):
        s, e = int(sp["start"]), int(sp["end"])
        if s < cursor or e > len(text):
            continue
        html_parts.append(html_lib.escape(text[cursor:s]).replace("\n", "<br>"))
        html_parts.append(
            f'<mark title="{html_lib.escape(str(sp["label"]))} ({float(sp["score"]):.2f})" '
            f'style="background:#ffe08a">{html_lib.escape(text[s:e])}'
            f'<sup style="font-size:9px;color:#666">{html_lib.escape(str(sp["label"]))}</sup></mark>'
        )
        cursor = e
    html_parts.append(html_lib.escape(text[cursor : cursor + 2500]).replace("\n", "<br>"))
    return HTML(
        "<div style='font-family:Georgia,serif;font-size:13px;line-height:1.45;"
        "max-height:420px;overflow:auto;border:1px solid #ddd;padding:12px'>"
        + "".join(html_parts)
        + "</div>"
    )

display(highlight(run_text, span_df if not span_df.empty else pd.DataFrame()))


## 5. Batch a few samples (optional)

Uses the same `FOLDER` filter + `LABELS` / backend from above. Keep `N` small on CPU.


In [ ]:
N = 3  # how many files from `candidates`
MAX_CHARS_BATCH = 8_000

rows = []
for _, r in candidates.head(N).iterrows():
    p = Path(r.html_path)
    if not p.exists():
        continue
    t = html_to_text(p.read_text(errors="replace"))[:MAX_CHARS_BATCH]
    try:
        sp = backend.predict(t, LABELS)
    except Exception as exc:  # noqa: BLE001
        print("fail", p.name, exc)
        continue
    labels_hit = sorted({s["label"] for s in sp})
    rows.append(
        {
            "entity_name": r.entity_name,
            "year": r.year,
            "file": p.name,
            "n_spans": len(sp),
            "labels": ", ".join(labels_hit[:8]) + ("…" if len(labels_hit) > 8 else ""),
            "top": max((s["score"] for s in sp), default=None),
        }
    )
    print(f"  {p.name}: {len(sp)} spans")

batch = pd.DataFrame(rows)
display(batch)


### Next steps

- Production path uses **segments** (not raw full-doc truncate) then ranks spans → Concise Rep.
- CLI: `python scripts/run_preprocess.py --path tender --stage all --cohort gold --limit 1 --skip-db`
- Inspect artifacts: `notebooks/preprocess_stage_inspect.ipynb`